In [3]:
#LLM finetuning
from huggingface_hub import login
login()
from typing_extensions import evaluate_forward_ref
#fine tuning pipeline
%pip install transformers datasets trl peft bitsandbytes accelerate evaluate
#load base_model
from transformers  import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

model_id= "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer= AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=False,
)

model= AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map= "auto"
)
#load the dataset
from datasets import load_dataset, DatasetDict

dataset_full = load_dataset("tatsu-lab/alpaca")
# Select first 1000 samples for the primary training set
dataset_train_1000 = dataset_full["train"].select(range(1000))

# Split the selected training subset into train and test for the trainer
train_test_splits = dataset_train_1000.train_test_split(test_size=0.1, seed=42) # Added seed for reproducibility

dataset = DatasetDict({
    "train": train_test_splits["train"],
    "test": train_test_splits["test"]
})

#set prompt format with instructions and input-output
def format_prompt(example):
    return {
        "text": f"""
        ### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""
    }

dataset = dataset.map(format_prompt)
#define lora_configuration
loraconfig= LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout= 0.05,
    bias= "none",
    task_type= "CAUSAL_LM"
)
model= get_peft_model(model, loraconfig)
#training with training arguments
training_args= TrainingArguments(
    per_device_train_batch_size= 2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate= 2e-4,
    fp16= False,
    bf16= False, # Explicitly disable bfloat16 to avoid conflict
    logging_steps= 10,
    save_strategy= "epoch",
    output_dir="./qlora-output" # Added output_dir
)

#trainer - instruction tuning
def formatting_func(example):
    return example["text"]
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    formatting_func=formatting_func,
    processing_class=tokenizer
)
#start training
trainer.train()
#save the model
trainer.save_model('./qlora-output')

#evaluate the fine tuned model
metrics= trainer.evaluate()
print(metrics)
#perplexity score calculation
import math
perplexity= math.exp(metrics["eval_loss"])
print(perplexity)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.940248
20,1.525764
30,1.249359
40,1.199370
50,1.168859
60,1.201190
70,1.164576
80,1.158809
90,1.179003
100,1.203599


{'eval_loss': 1.1943645477294922, 'eval_runtime': 6.6385, 'eval_samples_per_second': 15.064, 'eval_steps_per_second': 1.958}
3.3014591840620287


In [ ]:
#Test your Qlora finetuned model
from huggingface_hub import login
login()
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer= AutoTokenizer.from_pretrained(model_id)
model= AutoModelForCausalLM.from_pretrained(model_id)
model = PeftModel.from_pretrained(model,"./qlora-output")
#test with the sample prompt
prompt= "###Instruction:\n What is crop rotation?\n\n###Response:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=120)

print(tokenizer.decode(output[0], skip_special_tokens=True))


model.push_to_hub("NikhilRaman1203/tinyllama-alpaca-instruct-v1")
tokenizer.push_to_hub("NikhilRaman1203/tinyllama-alpaca-instruct-v1")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

###Instruction:
 What is crop rotation?

###Response:
 Crop rotation is the practice of planting different crops in the same field or area to prevent soil erosion, reduce the risk of pests and diseases, and improve soil fertility.



Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  14%|#3        | 1.23MB / 9.02MB            

CommitInfo(commit_url='https://huggingface.co/NikhilRaman1203/tinyllama-alpaca-instruct-v1/commit/a2b681ec6015f659770dec0b97a40114927c069d', commit_message='Upload tokenizer', commit_description='', oid='a2b681ec6015f659770dec0b97a40114927c069d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/NikhilRaman1203/tinyllama-alpaca-instruct-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='NikhilRaman1203/tinyllama-alpaca-instruct-v1'), pr_revision=None, pr_num=None)

In [8]:
#evaluation
!pip install evaluate
#Belu
from evaluate import load
predictions= [
    "Crop rotation helps improve soil fertility and reduce pests."
]

references = [
    [
        "Crop rotation improves soil fertility and reduces pests.",
        "Crop rotation is a farming practice that maintains soil health."
    ]
]
bleu= load("bleu")
results= bleu.compute(predictions=predictions, references=references)
print(results)

#Exact match score
exact_match= load("exact_match")
em_score= exact_match.compute(predictions=predictions, references=[references[0][0]])
print(em_score)

{'bleu': 0.0, 'precisions': [0.7, 0.4444444444444444, 0.125, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 1.1111111111111112, 'translation_length': 10, 'reference_length': 9}
{'exact_match': np.float64(0.0)}


In [11]:
import json

results = {
    "perplexity": 3.3,
    "bleu": 0.0,
    "exact_match": 0.0
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f)

In [12]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="evaluation_results.json",
    path_in_repo="evaluation_results.json",
    repo_id="NikhilRaman1203/tinyllama-alpaca-instruct-v1",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/NikhilRaman1203/tinyllama-alpaca-instruct-v1/commit/df096b7fd238959add889ec854d18119557f1ba5', commit_message='Upload evaluation_results.json with huggingface_hub', commit_description='', oid='df096b7fd238959add889ec854d18119557f1ba5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/NikhilRaman1203/tinyllama-alpaca-instruct-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='NikhilRaman1203/tinyllama-alpaca-instruct-v1'), pr_revision=None, pr_num=None)